# 逐手模型微調與推論

`aicup/train_permove.py` 微調 KataGo humanv0（10 份 policy head，每個等級一份），`aicup/infer_permove.py` 輸出每局 `pm`（目標側）與 `pmo`（對手側）。

- runtime：`smoke` 用 T4，其他用 A100
- 精度：`train_permove --amp auto`，A100 用 bf16，T4 用 float32；推論固定 float32
- 外部資源：KataGo `python/`（tag `v1.15.0`，MIT）與 `b18c384nbt-humanv0.ckpt`，由程式從官方網址下載
- Drive 的 `MyDrive/aicup-2026/` 要有：
  - `permove-features/spike/{train,holdout}/`（`smoke`、`train`）
  - `permove-features/val648/val/` 與 `humansl-analysis/val648-p3.jsonl`（`smoke`、`infer_val`）
  - `permove-features/test/val/` 與 `humansl-analysis/test-p3.jsonl`（`infer_test`）
- Colab Secrets 要有唯讀的 `GH_TOKEN`

Drive 的做法：

- 不跑 `aicup.colab_setup`。原因：它會讀 `rank-features/`，這裡用不到。
- 特徵先複製到 `/content`，訓練與推論期間不讀 Drive 上的特徵。
- checkpoint 直接寫在 Drive 的 `rank-trained-models/permove/<commit hash>/`。斷線時重開 runtime、依序執行所有 cell，會從 `last.pt` 接續。
- humanv0 的 checkpoint 快取在 Drive 的 `katago/`，下次不用重新下載。

## 參數

`MODE` 選 `smoke`、`train`、`infer_val`、`infer_test`。

- `smoke`：2,000 個局面訓練 30 步，再接續到 35 步，最後比較推論的 float32 與 bf16。重跑前先刪掉 `<hash>-smoke/`。
- `train`：spike 資料 1 個 epoch，checkpoint 在 `<hash>/`。
- `infer_val`：val648 兩側推論，輸出 `humansl-analysis/val648-pm.jsonl`。
- `infer_test`：測試集兩側推論，輸出 `humansl-analysis/test-pm.jsonl`。
- `BATCH`：訓練的 batch。T4 放不下 256 時改成 128，lr 不變。
- `TRAIN_HASH`：推論用哪一次訓練的 `model.pt`。`None` 時用目前的 commit。
- 換 `MODE` 時從本格起依序重跑。

In [ ]:
MODE = 'smoke'
BATCH = 256
TRAIN_HASH = None

DRIVE = '/content/drive/MyDrive/aicup-2026'
MODES = ('smoke', 'train', 'infer_val', 'infer_test')
assert MODE in MODES, '無效的 MODE：%s，只支援 %s' % (MODE, MODES)

## 取得程式、KataGo 與 humanv0

In [ ]:
import os, subprocess, shutil
from google.colab import drive, userdata

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'
KATAGO = '/content/KataGo'
KATAGO_COMMIT = 'c560d38585b446bc93b0a62c77364bc8e7d266a4'  # tag v1.15.0


def git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr.replace(userdata.get('GH_TOKEN'), '***'))
        raise SystemExit('git %s 失敗' % args[0])
    return r.stdout.strip()


if not os.path.exists(REPO):
    git('clone', '-q', 'https://%s@github.com/Jx-study/aicup-2026-go.git' % userdata.get('GH_TOKEN'), REPO)
os.chdir(REPO)
git('pull', '-q')
HASH = git('rev-parse', '--short', 'HEAD')
!pip install -q sgfmill
!git log --oneline -1
!nvidia-smi -L

if not os.path.exists(KATAGO + '/python/features.py'):
    git('clone', '-q', '--depth', '1', '--branch', 'v1.15.0', '--filter=blob:none', '--sparse',
        'https://github.com/lightvector/KataGo', KATAGO)
    git('-C', KATAGO, 'sparse-checkout', 'set', 'python')
assert git('-C', KATAGO, 'rev-parse', 'HEAD') == KATAGO_COMMIT, 'KataGo 的 commit 不是 v1.15.0'
os.environ['KATAGO_PYTHON'] = KATAGO + '/python'

from aicup import katago_torch as kt

HUMANV0 = '/content/models/b18c384nbt-humanv0.ckpt'
CACHE = DRIVE + '/katago/b18c384nbt-humanv0.ckpt'
os.makedirs(os.path.dirname(HUMANV0), exist_ok=True)
if not os.path.exists(HUMANV0) and os.path.exists(CACHE):
    shutil.copy(CACHE, HUMANV0)
kt.ensure_humanv0(HUMANV0)  # 沒有時下載，並檢查大小與 SHA-256
if not os.path.exists(CACHE):
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)
    shutil.copy(HUMANV0, CACHE)
print('commit %s，humanv0 %s' % (HASH, HUMANV0))

## 複製資料

依 `MODE` 只複製需要的特徵與 jsonl，最後列出分片數。

In [ ]:
NEEDS = {
    'smoke': ['permove-features/spike', 'permove-features/val648', 'humansl-analysis/val648-p3.jsonl'],
    'train': ['permove-features/spike'],
    'infer_val': ['permove-features/val648', 'humansl-analysis/val648-p3.jsonl'],
    'infer_test': ['permove-features/test', 'humansl-analysis/test-p3.jsonl'],
}
for rel in NEEDS[MODE]:
    src, dst = DRIVE + '/' + rel, '/content/' + rel
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(src, dst)
!find /content/permove-features -name '*.npz' | sed 's|/[^/]*$||' | sort | uniq -c

## 訓練或推論

- `smoke`：看 loss 是否下降、第二次執行是否印出「從 ... 接續，第 30 步」、每秒局面數，以及 float32 與 bf16 的每局特徵差最大值。
- `train`：log 同時寫到 checkpoint 目錄的 `train.log`。中途斷線時重跑本 cell 會接續。
- `infer_*`：推論只在最後寫檔，斷線時要整個重跑。
- 任何一個指令失敗（結束碼不是 0）都會停在本 cell，不會往下執行。

In [ ]:
CKPT_ROOT = DRIVE + '/rank-trained-models/permove'
TRAIN = 'python -m aicup.train_permove --data /content/permove-features/spike --humanv0 %s --batch %d' % (HUMANV0, BATCH)
INFER = {'infer_val': ('/content/permove-features/val648/val', 'val648-pm.jsonl'),
         'infer_test': ('/content/permove-features/test/val', 'test-pm.jsonl')}


def run(cmd, log=None):
    """執行指令並逐行印出，log 給定時同時附加寫入；結束碼不是 0 時停止。"""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    f = open(log, 'a') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if f:
            f.write(line)
            f.flush()
    if f:
        f.close()
    if p.wait():
        raise SystemExit('失敗（結束碼 %d）：%s' % (p.returncode, cmd))


if MODE == 'smoke':
    CKPT = '%s/%s-smoke' % (CKPT_ROOT, HASH)
    os.makedirs(CKPT, exist_ok=True)
    SMOKE = '%s --ckpt-dir %s --limit 2000' % (TRAIN, CKPT)
    run(SMOKE + ' --max-steps 30 --log-every 10', CKPT + '/train.log')
    run(SMOKE + ' --max-steps 35 --log-every 5', CKPT + '/train.log')
    run('python -m aicup.infer_permove --model %s/model.pt --data /content/permove-features/val648/val '
        '--compare-dtypes 20' % CKPT)
elif MODE == 'train':
    CKPT = '%s/%s' % (CKPT_ROOT, HASH)
    os.makedirs(CKPT, exist_ok=True)
    run('%s --ckpt-dir %s --log-every 200' % (TRAIN, CKPT), CKPT + '/train.log')
else:
    MODEL = '%s/%s/model.pt' % (CKPT_ROOT, TRAIN_HASH or HASH)
    DATA, OUT_NAME = INFER[MODE]
    run('python -m aicup.infer_permove --model %s --data %s --out /content/%s' % (MODEL, DATA, OUT_NAME),
        '/content/%s.log' % MODE)
    os.makedirs(DRIVE + '/humansl-analysis', exist_ok=True)
    for name in (OUT_NAME, MODE + '.log'):
        shutil.copy('/content/' + name, DRIVE + '/humansl-analysis/')

## 檢查

- `smoke`、`train`：列出 holdout 的 loss。前 30% 的步數內 policy loss 比第 0 步高超過 5% 時，停止並回報。原因：主幹的 lr 可能太大。
- `infer_*`：行數與 key 集合要和參考 jsonl 相同，每行都要有 `pm`、`pmo`。

In [ ]:
import json

if MODE in ('smoke', 'train'):
    hist = [json.loads(line) for line in open(CKPT + '/history.jsonl')]
    for h in hist:
        print('step %6d  policy %.4f  rank %.4f  acc %.3f' % (h['step'], h['policy_loss'], h['rank_loss'], h['rank_acc']))
    base, last = hist[0]['policy_loss'], hist[-1]['step']
    early = [h for h in hist if 0 < h['step'] <= 0.3 * last]
    if any(h['policy_loss'] > 1.05 * base for h in early):
        print('停止：前 30% 的步數內 holdout policy loss 比第 0 步高超過 5%')
else:
    ref = {'infer_val': 'val648-p3.jsonl', 'infer_test': 'test-p3.jsonl'}[MODE]
    want = {json.loads(line)['key'] for line in open('/content/humansl-analysis/' + ref)}
    rows = [json.loads(line) for line in open(DRIVE + '/humansl-analysis/' + OUT_NAME)]
    got = {r['key'] for r in rows}
    bad = [r['key'] for r in rows if r['pm'] is None or r['pmo'] is None]
    print('%d 行，key 和 %s 相同：%s，缺 pm 或 pmo：%d 局' % (len(rows), ref, got == want and len(rows) == len(want), len(bad)))

## 釋放 runtime

跑完才執行。程式結束後 runtime 不會自動中斷，閒置也會扣額度。

In [ ]:
from google.colab import drive, runtime

drive.flush_and_unmount()
runtime.unassign()